In [ ]:
import time, numpy as np, pandas as pd
from google.colab import drive
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import f1_score

drive.mount('/content/drive')
D = "/content/drive/MyDrive/phd_data/"

In [ ]:
import os, glob
D = "/content/drive/MyDrive/phd_data/"
print(sorted(os.listdir(D)))
cands = [f for f in os.listdir(D) if f.lower().endswith(".csv") and "train" in f.lower()]
print("candidates:", cands)

In [ ]:
import time, numpy as np, pandas as pd
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import f1_score

D = "/content/drive/MyDrive/phd_data/"
cands = [f for f in os.listdir(D) if f.lower().endswith(".csv") and "train" in f.lower()]
assert len(cands) == 1, f"expected exactly one train CSV, found: {cands}"
tr_df = pd.read_csv(D + cands[0])
print("using", cands[0], tr_df.shape)

ytr = LabelEncoder().fit_transform(tr_df["attack_cat"].astype(str).str.strip())
X = tr_df.drop(columns=[c for c in ["attack_cat", "label", "id"] if c in tr_df.columns]).copy()
cat = [c for c in ["proto", "service", "state"] if c in X.columns]
X[cat] = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit_transform(X[cat].astype(str))
Xtr = X
fold_id = np.load(D + "fold_id.npy")
assert len(Xtr) == len(ytr) == len(fold_id) == 175341, (len(Xtr), len(ytr), len(fold_id))
print("loaded", Xtr.shape, "classes", len(np.unique(ytr)))

configs = [(8,0.10,300),(6,0.10,300),(10,0.10,300),(12,0.10,300),(16,0.10,300),(8,0.05,600),(8,0.20,150)]
rows = []
for md, lr, ne in configs:
    t0, f1s = time.time(), []
    for k in [0, 1, 2]:
        a = np.where(fold_id != k)[0]; b = np.where(fold_id == k)[0]
        m = XGBClassifier(objective="multi:softprob", n_estimators=ne, learning_rate=lr,
                          max_depth=md, tree_method="hist", device="cuda", random_state=0)
        m.fit(Xtr.iloc[a], ytr[a])
        f1s.append(f1_score(ytr[b], m.predict(Xtr.iloc[b]), average="macro"))
    rows.append((md, lr, ne, round(np.mean(f1s), 4), round(time.time() - t0)))
    print(rows[-1])

res = pd.DataFrame(rows, columns=["depth", "lr", "trees", "mean macro-F1", "seconds"])
print(res.to_string(index=False))
print("\nbest minus current:", round(res["mean macro-F1"].max() - res.iloc[0]["mean macro-F1"], 4))

In [ ]:
import os, time, shutil, numpy as np, pandas as pd
from google.colab import drive
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import f1_score

drive.mount('/content/drive')
D = "/content/drive/MyDrive/phd_data/"
dev = "cuda" if shutil.which("nvidia-smi") else "cpu"
print("device:", dev)

tr_df = pd.read_csv(D + "UNSW_NB15_training-set (1).csv")
ytr = LabelEncoder().fit_transform(tr_df["attack_cat"].astype(str).str.strip())
X = tr_df.drop(columns=[c for c in ["attack_cat", "label", "id"] if c in tr_df.columns]).copy()
cat = [c for c in ["proto", "service", "state"] if c in X.columns]
X[cat] = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit_transform(X[cat].astype(str))
Xtr = X
fold_id = np.load(D + "fold_id.npy")
assert len(Xtr) == len(ytr) == len(fold_id) == 175341

# sanity: old OOF on folds 0-2 should be near 0.6101
old = np.load(D + "oof_xgb.npy")
mf = lambda P, ks: f1_score(ytr[np.isin(fold_id, list(ks))], P[np.isin(fold_id, list(ks))].argmax(1), average="macro")
print("old OOF, folds 0-2:", round(mf(old, [0, 1, 2]), 4), "(expect about 0.61)")

new = np.zeros_like(old)
for k in range(10):
    f = D + f"oof_d16_fold{k}.npy"
    b = np.where(fold_id == k)[0]
    if os.path.exists(f):
        new[b] = np.load(f); print("fold", k, "loaded from Drive"); continue
    t0 = time.time()
    a = np.where(fold_id != k)[0]
    m = XGBClassifier(objective="multi:softprob", n_estimators=300, learning_rate=0.1, max_depth=16,
                      tree_method="hist", device=dev, random_state=0)
    m.fit(Xtr.iloc[a], ytr[a])
    new[b] = m.predict_proba(Xtr.iloc[b])
    np.save(f, new[b])
    print("fold", k, "done", round(time.time() - t0), "s")

np.save(D + "oof_xgb_d16.npy", new)
print("\nALL folds   old:", round(mf(old, range(10)), 4), " new:", round(mf(new, range(10)), 4))
print("folds 0-2   old:", round(mf(old, [0, 1, 2]), 4), " new:", round(mf(new, [0, 1, 2]), 4), " (tuned on these)")
print("folds 3-9   old:", round(mf(old, range(3, 10)), 4), " new:", round(mf(new, range(3, 10)), 4), " (fair test)")
print("gain on folds 3-9:", round(mf(new, range(3, 10)) - mf(old, range(3, 10)), 4))

In [ ]:
import os, shutil, numpy as np, pandas as pd
from google.colab import drive
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.metrics import f1_score

drive.mount('/content/drive')
D = "/content/drive/MyDrive/phd_data/"
dev = "cuda" if shutil.which("nvidia-smi") else "cpu"
print("device:", dev)

tr_df = pd.read_csv(D + "UNSW_NB15_training-set (1).csv")
te_df = pd.read_csv(D + "UNSW_NB15_testing-set (1).csv")
le = LabelEncoder().fit(tr_df["attack_cat"].astype(str).str.strip())
ytr = le.transform(tr_df["attack_cat"].astype(str).str.strip())
yte = le.transform(te_df["attack_cat"].astype(str).str.strip())

drop = [c for c in ["attack_cat", "label", "id"] if c in tr_df.columns]
Xtr = tr_df.drop(columns=drop).copy(); Xte = te_df.drop(columns=drop).copy()
cat = [c for c in ["proto", "service", "state"] if c in Xtr.columns]
oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1).fit(Xtr[cat].astype(str))
Xtr[cat] = oe.transform(Xtr[cat].astype(str)); Xte[cat] = oe.transform(Xte[cat].astype(str))
print("train", Xtr.shape, "test", Xte.shape, "classes", list(le.classes_))

# check labels/class order against the earlier saved test probabilities
p_old = np.load(D + "unsw_test_p_x_s0.npy")
assert p_old.shape == (len(Xte), 10), p_old.shape
print("old XGB test macro-F1 (seed 0):", round(f1_score(yte, p_old.argmax(1), average="macro"), 4), "(expect about 0.505)")

m = XGBClassifier(objective="multi:softprob", n_estimators=300, learning_rate=0.1, max_depth=16,
                  tree_method="hist", device=dev, random_state=0)
m.fit(Xtr, ytr)
p_new = m.predict_proba(Xte)
np.save(D + "unsw_test_p_x_d16.npy", p_new)
print("NEW XGB depth16 test macro-F1:", round(f1_score(yte, p_new.argmax(1), average="macro"), 4))

nrm = list(le.classes_).index("Normal")
for name, p in [("old", p_old), ("new", p_new)]:
    pred = p.argmax(1)
    fpr = np.mean(pred[yte == nrm] != nrm)
    det = np.mean(pred[yte != nrm] != nrm)
    print(name, "false alarm rate:", round(fpr, 4), " attack detection rate:", round(det, 4))

In [ ]:
import numpy as np, pandas as pd, warnings
from google.colab import drive
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore")
drive.mount('/content/drive')
D = "/content/drive/MyDrive/phd_data/"

tr_df = pd.read_csv(D + "UNSW_NB15_training-set (1).csv")
te_df = pd.read_csv(D + "UNSW_NB15_testing-set (1).csv")
le = LabelEncoder().fit(tr_df["attack_cat"].astype(str).str.strip())
ytr = le.transform(tr_df["attack_cat"].astype(str).str.strip())
yte = le.transform(te_df["attack_cat"].astype(str).str.strip())
fold_id = np.load(D + "fold_id.npy"); nrm = list(le.classes_).index("Normal")
S = [0, 1, 2]

oof = {"xgb_old": np.load(D + "oof_xgb.npy"), "xgb_new": np.load(D + "oof_xgb_d16.npy"),
       "rf": np.load(D + "oof_rf.npy"), "cnn": np.load(D + "oof_cnn.npy")}
tst = {"xgb_old": [np.load(D + f"unsw_test_p_x_s{s}.npy") for s in S],
       "xgb_new": [np.load(D + "unsw_test_p_x_d16.npy")] * 3,
       "rf": [np.load(D + f"unsw_test_p_r_s{s}.npy") for s in S],
       "cnn": [np.load(D + f"unsw_test_p_c_s{s}.npy") for s in S]}

mf = lambda y, p: f1_score(y, p, average="macro")
fpr = lambda y, p: float(np.mean(p[y == nrm] != nrm))

def score(name, parts):
    """parts: list of keys. One key + 'argmax' => base model; else LR on concatenated probs."""
    Ftr = np.hstack([oof[k] for k in parts])
    Fte = [np.hstack([tst[k][s] for k in parts]) for s in S]
    if name.startswith("BASE"):
        cv = mf(ytr, Ftr.argmax(1)); preds = [f.argmax(1) for f in Fte]
    else:
        pr = np.zeros(len(ytr), int)
        for k in range(10):
            a, b = fold_id != k, fold_id == k
            pr[b] = LogisticRegression(max_iter=1000).fit(Ftr[a], ytr[a]).predict(Ftr[b])
        cv = mf(ytr, pr)
        m = LogisticRegression(max_iter=1000).fit(Ftr, ytr); preds = [m.predict(f) for f in Fte]
    t = [mf(yte, p) for p in preds]; f = [fpr(yte, p) for p in preds]
    row = (name, round(cv, 4), round(np.mean(t), 4), round(np.std(t), 4), round(np.mean(f), 4))
    print(row); return row

rows = []
for tag, x in [("OLD", "xgb_old"), ("NEW", "xgb_new")]:
    rows += [score(f"{tag} BASE XGB", [x]),
             score(f"{tag} LR on XGB", [x]),
             score(f"{tag} Stack (XGB+RF)", [x, "rf"]),
             score(f"{tag} Stack+CNN", [x, "rf", "cnn"])]
rows += [score("BASE RF", ["rf"]), score("BASE CNN", ["cnn"])]
res = pd.DataFrame(rows, columns=["system", "CV macro-F1", "test mean", "test sd", "test FPR"])
res.to_csv(D + "unsw_rebuild_table.csv", index=False)
print("\n", res.to_string(index=False))

In [ ]:
import numpy as np, pandas as pd, warnings
from google.colab import drive
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore")
drive.mount('/content/drive')
D = "/content/drive/MyDrive/phd_data/"

tr_df = pd.read_csv(D + "UNSW_NB15_training-set (1).csv")
te_df = pd.read_csv(D + "UNSW_NB15_testing-set (1).csv")
le = LabelEncoder().fit(tr_df["attack_cat"].astype(str).str.strip())
ytr = le.transform(tr_df["attack_cat"].astype(str).str.strip())
yte = le.transform(te_df["attack_cat"].astype(str).str.strip())
fold_id = np.load(D + "fold_id.npy"); nrm = list(le.classes_).index("Normal"); S = [0, 1, 2]

oof = {"xgb": np.load(D + "oof_xgb_d16.npy"), "rf": np.load(D + "oof_rf.npy"), "cnn": np.load(D + "oof_cnn.npy")}
tst = {"xgb": [np.load(D + "unsw_test_p_x_d16.npy")] * 3,
       "rf": [np.load(D + f"unsw_test_p_r_s{s}.npy") for s in S],
       "cnn": [np.load(D + f"unsw_test_p_c_s{s}.npy") for s in S]}
lg = lambda p: np.log(np.clip(p, 1e-6, 1.0))
mf = lambda y, p: f1_score(y, p, average="macro")
fpr = lambda y, p: float(np.mean(p[y == nrm] != nrm))

def raw(name, k):
    t = [mf(yte, p.argmax(1)) for p in tst[k]]; f = [fpr(yte, p.argmax(1)) for p in tst[k]]
    row = (name, round(mf(ytr, oof[k].argmax(1)), 4), round(np.mean(t), 4), round(np.mean(f), 4)); print(row); return row

def logit_stack(name, parts):
    Ftr = np.hstack([lg(oof[k]) for k in parts]); Fte = [np.hstack([lg(tst[k][s]) for k in parts]) for s in S]
    mk = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))
    pr = np.zeros(len(ytr), int)
    for k in range(10):
        a, b = fold_id != k, fold_id == k
        pr[b] = mk().fit(Ftr[a], ytr[a]).predict(Ftr[b])
    m = mk().fit(Ftr, ytr); preds = [m.predict(f) for f in Fte]
    t = [mf(yte, p) for p in preds]; f = [fpr(yte, p) for p in preds]
    row = (name, round(mf(ytr, pr), 4), round(np.mean(t), 4), round(np.mean(f), 4)); print(row); return row

rows = [raw("RAW XGB depth16", "xgb"), raw("RAW RF", "rf"), raw("RAW CNN", "cnn"),
        logit_stack("LOGIT LR on XGB", ["xgb"]),
        logit_stack("LOGIT Stack (XGB+RF)", ["xgb", "rf"]),
        logit_stack("LOGIT Stack+CNN", ["xgb", "rf", "cnn"])]
res = pd.DataFrame(rows, columns=["system", "CV macro-F1", "test mean", "test FPR"])
res.to_csv(D + "unsw_logit_check.csv", index=False)
print("\n", res.to_string(index=False))

In [ ]:
import numpy as np, pandas as pd, warnings
from google.colab import drive
from scipy.stats import binomtest
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore")
drive.mount('/content/drive')
D = "/content/drive/MyDrive/phd_data/"
K, S = 10, [0, 1, 2]

tr_df = pd.read_csv(D + "UNSW_NB15_training-set (1).csv")
te_df = pd.read_csv(D + "UNSW_NB15_testing-set (1).csv")
le = LabelEncoder().fit(tr_df["attack_cat"].astype(str).str.strip())
ytr = le.transform(tr_df["attack_cat"].astype(str).str.strip())
yte = le.transform(te_df["attack_cat"].astype(str).str.strip())
fold_id = np.load(D + "fold_id.npy"); nrm = list(le.classes_).index("Normal")

feat = [c for c in tr_df.columns if c not in ("id", "attack_cat", "label")]
gtr = pd.factorize(pd.util.hash_pandas_object(tr_df[feat], index=False).values)[0]
gte = pd.factorize(pd.util.hash_pandas_object(te_df[feat], index=False).values)[0]
assert pd.Series(fold_id).groupby(gtr).nunique().max() == 1, "groups do not match fold_id"
print("train groups", gtr.max() + 1, "test groups", gte.max() + 1)

oof = {"xgb": np.load(D + "oof_xgb_d16.npy"), "rf": np.load(D + "oof_rf.npy"), "cnn": np.load(D + "oof_cnn.npy")}
tst = {"xgb": [np.load(D + "unsw_test_p_x_d16.npy")] * 3,
       "rf": [np.load(D + f"unsw_test_p_r_s{s}.npy") for s in S],
       "cnn": [np.load(D + f"unsw_test_p_c_s{s}.npy") for s in S]}
lg = lambda p: np.log(np.clip(p, 1e-6, 1.0))

def build(parts, logit):
    f = lg if logit else (lambda p: p)
    Ftr = np.hstack([f(oof[k]) for k in parts]); Fte = [np.hstack([f(tst[k][s]) for k in parts]) for s in S]
    mk = (lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000))) if logit \
         else (lambda: LogisticRegression(max_iter=2000))
    cvp = np.zeros((len(ytr), K))
    for k in range(10):
        a, b = fold_id != k, fold_id == k
        cvp[b] = mk().fit(Ftr[a], ytr[a]).predict_proba(Ftr[b])
    m = mk().fit(Ftr, ytr)
    return cvp, [m.predict_proba(x) for x in Fte]

cv_p = {"XGB": oof["xgb"], "RF": oof["rf"], "CNN": oof["cnn"]}
te_p = {"XGB": tst["xgb"], "RF": tst["rf"], "CNN": tst["cnn"]}
for nm, parts, lgt in [("Stack", ["xgb", "rf"], False), ("Stack+CNN", ["xgb", "rf", "cnn"], False),
                       ("Stack (log)", ["xgb", "rf"], True), ("Stack+CNN (log)", ["xgb", "rf", "cnn"], True)]:
    cv_p[nm], te_p[nm] = build(parts, lgt); print("built", nm)
np.savez_compressed(D + "unsw_final_stacks.npz",
    **{f"cv_{k}": v for k, v in cv_p.items()}, **{f"te_{k}_s{s}": te_p[k][s] for k in te_p for s in S})

# ---------- grouped bootstrap ----------
def gcounts(pred, y, gid):
    G = gid.max() + 1; ok = pred == y
    tp = np.bincount(gid[ok] * K + y[ok], minlength=G * K).reshape(G, K)
    pc = np.bincount(gid * K + pred, minlength=G * K).reshape(G, K)
    tc = np.bincount(gid * K + y, minlength=G * K).reshape(G, K)
    return tp, pc, tc
def boot(preds, y, gid, B=1000, seed=0):
    G = gid.max() + 1; cnt = {k: gcounts(v, y, gid) for k, v in preds.items()}
    rng = np.random.default_rng(seed); out = {k: np.zeros(B) for k in preds}
    for b in range(B):
        w = np.bincount(rng.integers(0, G, G), minlength=G).astype(float)
        for k, (tp, pc, tc) in cnt.items():
            TP, PC, TC = w @ tp, w @ pc, w @ tc
            out[k][b] = (2 * TP / np.maximum(PC + TC, 1))[TC > 0].mean()
    return out
stacks = ["Stack", "Stack+CNN", "Stack (log)", "Stack+CNN (log)"]
def report_boot(title, preds, y, gid):
    r = boot(preds, y, gid); print(f"\n== {title} (grouped bootstrap, 1000) ==")
    for k in preds: print(f"{k:16s} F1 {r[k].mean():.4f}  CI [{np.percentile(r[k],2.5):.4f}, {np.percentile(r[k],97.5):.4f}]")
    for s in stacks:
        d = r[s] - r["XGB"]; lo, hi = np.percentile(d, [2.5, 97.5])
        print(f"  {s} - XGB: {d.mean():+.4f}  CI [{lo:+.4f}, {hi:+.4f}]  {'excludes 0' if lo > 0 or hi < 0 else 'includes 0'}")
names = ["XGB"] + stacks
report_boot("CV", {k: cv_p[k].argmax(1) for k in names}, ytr, gtr)
report_boot("TEST seed 0", {k: te_p[k][0].argmax(1) for k in names}, yte, gte)

# ---------- McNemar (per-row accuracy), Holm ----------
print("\n== McNemar on test accuracy, seed 0 (per-row accuracy, not macro-F1) ==")
px = te_p["XGB"][0].argmax(1) == yte; pv = []
for s in stacks:
    ps = te_p[s][0].argmax(1) == yte
    b, c = int((px & ~ps).sum()), int((~px & ps).sum())
    p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0; pv.append((s, b, c, p))
order = sorted(range(len(pv)), key=lambda i: pv[i][3]); adj = {}; run = 0
for r, i in enumerate(order):
    run = max(run, min(1.0, pv[i][3] * (len(pv) - r))); adj[i] = run
for i, (s, b, c, p) in enumerate(pv): print(f"{s:16s} XGB-only-right {b:6d}  stack-only-right {c:6d}  p {p:.4g}  Holm {adj[i]:.4g}")

# ---------- Normal-weight sweep, test, 3 seeds ----------
print("\n== Normal-weight sweep (test, 3-seed mean): macro-F1 / false alarm / attack detection ==")
for nm in ["XGB", "RF", "Stack", "Stack+CNN"]:
    for w in [1, 2, 4, 8]:
        r = []
        for s in S:
            Q = te_p[nm][s].copy(); Q[:, nrm] *= w; p = Q.argmax(1)
            r.append((f1_score(yte, p, average="macro"), np.mean(p[yte == nrm] != nrm), np.mean(p[yte != nrm] != nrm)))
        r = np.mean(r, 0); print(f"{nm:10s} w={w}  F1 {r[0]:.4f}  FPR {r[1]:.3f}  det {r[2]:.3f}")

# ---------- per-class F1, test, 3-seed mean ----------
print("\n== Per-class F1 (test, 3-seed mean) ==")
rows = {nm: np.mean([f1_score(yte, te_p[nm][s].argmax(1), average=None, labels=range(K)) for s in S], 0)
        for nm in ["XGB", "RF", "CNN", "Stack", "Stack+CNN"]}
pc = pd.DataFrame(rows, index=le.classes_).round(3); print(pc.to_string())
pc.to_csv(D + "unsw_final_perclass.csv")

In [ ]:
import numpy as np, pandas as pd, warnings
from google.colab import drive
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, recall_score
warnings.filterwarnings("ignore")
drive.mount('/content/drive')
D = "/content/drive/MyDrive/phd_data/"
tr_df = pd.read_csv(D + "UNSW_NB15_training-set (1).csv")
te_df = pd.read_csv(D + "UNSW_NB15_testing-set (1).csv")
le = LabelEncoder().fit(tr_df["attack_cat"].astype(str).str.strip())
ytr = le.transform(tr_df["attack_cat"].astype(str).str.strip())
yte = le.transform(te_df["attack_cat"].astype(str).str.strip())
fold_id = np.load(D + "fold_id.npy"); nrm = list(le.classes_).index("Normal"); K = 10
fpr = lambda y, p: float(np.mean(p[y == nrm] != nrm))
mf = lambda y, p: f1_score(y, p, average="macro")
lg = lambda p: np.log(np.clip(p, 1e-6, 1.0))

Z = np.load(D + "unsw_final_stacks.npz")
cv = {k[3:]: Z[k] for k in Z.files if k.startswith("cv_")}
print("== CV macro-F1 and false alarm rate (Table 1) ==")
for k in ["XGB", "RF", "CNN", "Stack", "Stack+CNN", "Stack (log)", "Stack+CNN (log)"]:
    p = cv[k].argmax(1); print(f"{k:18s} F1 {mf(ytr,p):.4f}  FPR {fpr(ytr,p):.4f}")
oofx = cv["XGB"]
for name, F, mk in [("LR on XGB", oofx, lambda: LogisticRegression(max_iter=2000)),
                    ("LR on XGB (log)", lg(oofx), lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)))]:
    pr = np.zeros(len(ytr), int)
    for k in range(10):
        a, b = fold_id != k, fold_id == k
        pr[b] = mk().fit(F[a], ytr[a]).predict(F[b])
    print(f"{name:18s} F1 {mf(ytr,pr):.4f}  FPR {fpr(ytr,pr):.4f}")

print("\n== Out-of-fold weight sweep, depth-16 XGBoost ==")
ws = [0.5, 0.75, 1, 1.25, 1.5, 2, 3, 4, 6, 8]; res = []
for w in ws:
    Q = oofx.copy(); Q[:, nrm] *= w; p = Q.argmax(1)
    res.append((w, mf(ytr, p), fpr(ytr, p))); print(f"OOF w={w}  F1 {res[-1][1]:.4f}  FPR {res[-1][2]:.4f}")
best = max(res, key=lambda r: r[1])[0]
pt = np.load(D + "unsw_test_p_x_d16.npy")
Q = pt.copy(); Q[:, nrm] *= best
print("best OOF weight:", best, "-> test macro-F1", round(mf(yte, Q.argmax(1)), 4), "test FPR", round(fpr(yte, Q.argmax(1)), 4))

print("\n== Test: depth-16 XGBoost per class at each weight ==")
print(list(le.classes_))
for w in [1, 2, 4, 8]:
    Q = pt.copy(); Q[:, nrm] *= w; p = Q.argmax(1)
    print(f"w={w}  recall", np.round(recall_score(yte, p, average=None, labels=range(K)), 3))
    print(f"      F1    ", np.round(f1_score(yte, p, average=None, labels=range(K)), 3))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---------- Figure 1: CV macro-F1 vs official-test macro-F1 ----------
unsw = {  # system: (CV, test)
 "XGBoost":(0.626,0.510), "RF":(0.583,0.475), "CNN":(0.437,0.397),
 "LR on XGB":(0.616,0.503), "Stack RF+XGB":(0.619,0.504), "Stack RF+XGB+CNN":(0.619,0.504)}
nsl = {
 "XGBoost":(0.944,0.521), "RF":(0.926,0.489), "CNN":(0.769,0.463),
 "LR on XGB":(0.941,0.510), "Stack RF+XGB":(0.927,0.511), "Stack RF+XGB+CNN":(0.946,0.516)}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, d, title in zip(axes, [unsw, nsl], ["UNSW-NB15", "NSL-KDD"]):
    names = list(d); x = np.arange(len(names)); w = 0.38
    ax.bar(x-w/2, [d[n][0] for n in names], w, label="Grouped CV", color="#4C72B0")
    ax.bar(x+w/2, [d[n][1] for n in names], w, label="Official test", color="#DD8452")
    for i, n in enumerate(names):
        ax.text(i, max(d[n])+0.02, f"-{d[n][0]-d[n][1]:.2f}", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=35, ha="right", fontsize=8)
    ax.set_title(title); ax.set_ylim(0, 1.05)
axes[0].set_ylabel("Macro-F1"); axes[0].legend(frameon=False)
plt.tight_layout(); plt.savefig("fig1_cv_vs_test.png", dpi=300); plt.show()

# ---------- Figure 2: false alarm rate vs attack detection (UNSW, weight sweep) ----------
# (FPR, detection) at Normal weight w = 1, 2, 4, 8
sweep = {
 "XGBoost":        [(0.246,0.973),(0.197,0.962),(0.153,0.951),(0.118,0.938)],
 "RF":             [(0.244,0.978),(0.136,0.933),(0.068,0.883),(0.034,0.839)],
 "Stack RF+XGB":   [(0.224,0.973),(0.175,0.955),(0.126,0.926),(0.080,0.900)],
 "Stack RF+XGB+CNN":[(0.237,0.974),(0.181,0.960),(0.124,0.936),(0.071,0.901)]}
fig, ax = plt.subplots(figsize=(6, 4.6))
for (name, pts), c in zip(sweep.items(), ["#4C72B0","#55A868","#C44E52","#8172B2"]):
    xs, ys = zip(*pts)
    ax.plot(xs, ys, "-o", label=name, color=c)
    for (xx, yy), wt in zip(pts, [1,2,4,8]):
        ax.annotate(f"w={wt}", (xx, yy), textcoords="offset points", xytext=(4,4), fontsize=7)
ax.set_xlabel("False alarm rate (lower is better)")
ax.set_ylabel("Attack detection (higher is better)")
ax.set_title("UNSW-NB15 test: Normal-weight sweep")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("fig2_far_vs_detection.png", dpi=300); plt.show()

In [ ]:
import numpy as np, pandas as pd, glob, os
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from google.colab import drive
drive.mount('/content/drive')
D = "/content/drive/MyDrive/phd_data/"

# test labels (auto-detect the test CSV, skip the "(1)" duplicates only if a clean one exists)
cands = [f for f in os.listdir(D) if "test" in f.lower() and f.endswith(".csv") and "unsw" in f.lower()]
print("test csv candidates:", cands)
te = pd.read_csv(D + cands[0])
classes = sorted(te["attack_cat"].str.strip().unique())
print(classes)                       # Normal must be at index 6
y = te["attack_cat"].str.strip().map({c:i for i,c in enumerate(classes)}).values
NORMAL = classes.index("Normal")

p = np.load(D + "unsw_test_p_x_d16.npy")   # depth-16 XGBoost test probabilities
def point(w):
    q = p.copy(); q[:, NORMAL] *= w
    pred = q.argmax(1)
    fpr = (pred[y == NORMAL] != NORMAL).mean()
    det = (pred[y != NORMAL] != NORMAL).mean()
    return f1_score(y, pred, average="macro"), fpr, det

ws = [1, 2, 4, 8, 12, 16, 32]
rows = [(w, *point(w)) for w in ws]
print(pd.DataFrame(rows, columns=["w", "macroF1", "FPR", "detection"]).round(4).to_string(index=False))
# CHECK: w=1 must print macroF1 0.5102, FPR 0.2461, detection 0.9727. If not, stop and paste the output.

# ---- redraw Figure 2 ----
sweep = {
 "XGBoost":   [(r[2], r[3]) for r in rows],
 "RF":        [(0.244,0.978),(0.136,0.933),(0.068,0.883),(0.034,0.839)],
 "Stack RF+XGB":[(0.224,0.973),(0.175,0.955),(0.126,0.926),(0.080,0.900)],
 "Stack RF+XGB+CNN":[(0.237,0.974),(0.181,0.960),(0.124,0.936),(0.071,0.901)]}
wlab = {"XGBoost": ws, "RF":[1,2,4,8], "Stack RF+XGB":[1,2,4,8], "Stack RF+XGB+CNN":[1,2,4,8]}
fig, ax = plt.subplots(figsize=(6.4, 4.8))
for (name, pts), c in zip(sweep.items(), ["#4C72B0","#55A868","#C44E52","#8172B2"]):
    xs, ys = zip(*pts); ax.plot(xs, ys, "-o", ms=4, label=name, color=c)
    for (xx, yy), wt in zip(pts, wlab[name]):
        if wt in (1, wlab[name][-1]):
            ax.annotate(f"w={wt}", (xx, yy), textcoords="offset points", xytext=(5,-10), fontsize=7, color=c)
ax.set_xlabel("False alarm rate (lower is better)")
ax.set_ylabel("Attack detection (higher is better)")
ax.set_title("UNSW-NB15 test: Normal-weight sweep (diagnostic, not tuned)")
ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.savefig("fig2_far_vs_detection.png", dpi=300); plt.show()

In [ ]:
import os, numpy as np
from google.colab import drive
drive.mount('/content/drive')
D = "/content/drive/MyDrive/phd_data/"
for f in sorted(os.listdir(D)):
    if any(k in f.lower() for k in ["nsl", "kdd", "fold"]):
        p = D + f
        if f.endswith(".npy"):
            a = np.load(p, mmap_mode="r"); print(f, a.shape, a.dtype)
        else:
            print(f, os.path.getsize(p))

In [ ]:
import numpy as np, pandas as pd, shutil, time
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from google.colab import drive
drive.mount('/content/drive')
D = "/content/drive/MyDrive/phd_data/"

# ---- data (standard 5-family mapping, httptunnel -> U2R) ----
fam = {}
for n in "back land neptune pod smurf teardrop apache2 udpstorm processtable worm mailbomb".split(): fam[n]="DoS"
for n in "ipsweep nmap portsweep satan mscan saint".split(): fam[n]="Probe"
for n in ("ftp_write guess_passwd imap multihop phf spy warezclient warezmaster sendmail named "
          "snmpgetattack snmpguess xlock xsnoop").split(): fam[n]="R2L"
for n in "buffer_overflow loadmodule perl rootkit httptunnel ps sqlattack xterm".split(): fam[n]="U2R"
fam["normal"]="Normal"
classes = ["DoS","Normal","Probe","R2L","U2R"]
cid = {c:i for i,c in enumerate(classes)}

tr = pd.read_csv(D+"KDDTrain+.txt", header=None)
y = tr[41].str.strip().map(fam).map(cid).values
X = tr.drop(columns=[41,42]); X.columns = range(X.shape[1])
fold = np.load(D+"nsl_fold_s0.npy")
assert len(X)==len(y)==len(fold), (len(X), len(y), len(fold))
print("class counts:", dict(zip(classes, np.bincount(y))))   # expect U2R 52, R2L 995
print("folds:", np.bincount(fold))

CAT = [1,2,3]
def prep():
    return ColumnTransformer([("oh", OneHotEncoder(handle_unknown="ignore"), CAT)],
                             remainder="passthrough", sparse_threshold=0)
dev = "cuda" if shutil.which("nvidia-smi") else "cpu"
print("device:", dev)

def run(depth, folds):
    """train on the other folds, predict the held-out fold; pooled macro-F1 over `folds`"""
    yt, yp = [], []
    for k in folds:
        te = fold==k; trn = ~te
        pp = prep().fit(X[trn])                       # preprocessing fitted on training rows only
        m = XGBClassifier(objective="multi:softprob", n_estimators=300, learning_rate=0.1,
                          max_depth=depth, tree_method="hist", device=dev, random_state=0)
        m.fit(pp.transform(X[trn]), y[trn])
        yp.append(m.predict(pp.transform(X[te]))); yt.append(y[te])
    yt, yp = np.concatenate(yt), np.concatenate(yp)
    return f1_score(yt, yp, average="macro"), f1_score(yt, yp, average=None, labels=range(5))

t0 = time.time()
depths = [4, 6, 8, 10, 12, 16, 20]
res = {}
for d in depths:
    res[d] = run(d, [0,1,2]); print(f"depth {d:>2}: folds 0-2 macro-F1 {res[d][0]:.4f}   ({time.time()-t0:.0f}s)")

best = max(depths, key=lambda d: res[d][0])
print("\nbest on folds 0-2:", best, " | current (8):", round(res[8][0],4), " | gain:", round(res[best][0]-res[8][0],4))

# ---- confirm on unseen folds 3-9 ----
rest = range(3,10)
f_best, pc_best = run(best, rest)
f_cur,  pc_cur  = run(8, rest)
print(f"\nfolds 3-9: depth {best} {f_best:.4f} vs depth 8 {f_cur:.4f}   gain {f_best-f_cur:+.4f}")
print("per-class F1 (DoS,Normal,Probe,R2L,U2R)")
print("  depth", best, np.round(pc_best,3)); print("  depth 8", np.round(pc_cur,3))
pd.DataFrame({"depth":depths,"folds0_2_macroF1":[res[d][0] for d in depths]}).to_csv(D+"nsl_tuning_check.csv", index=False)